# Laboratorio 3 — Análisis de Ventas (Customer / Invoice)

Dataset: `Customer.csv` (59 clientes) e `Invoice.csv` (412 facturas).
Cada punto del enunciado está resuelto en su propia sección.

## 1. Carga de los archivos

Ambos CSV están en la misma carpeta que este notebook, así que se leen con
`pd.read_csv()` usando la ruta relativa.

In [ ]:
import pandas as pd

customer = pd.read_csv("Customer.csv")
invoice  = pd.read_csv("Invoice.csv")

print("Customer:", customer.shape)
print("Invoice :", invoice.shape)
customer.head()

In [ ]:
invoice.head()

## 2. Nulos y duplicados

Revisamos cuántos valores faltantes hay por columna y si existen filas repetidas.

In [ ]:
print("--- Nulos en Customer ---")
print(customer.isnull().sum())

print("\n--- Nulos en Invoice ---")
print(invoice.isnull().sum())

In [ ]:
print("Duplicados en Customer:", customer.duplicated().sum())
print("Duplicados en Invoice :", invoice.duplicated().sum())

# Duplicados por la llave primaria (debe ser 0)
print("CustomerId repetidos:", customer["CustomerId"].duplicated().sum())
print("InvoiceId repetidos :", invoice["InvoiceId"].duplicated().sum())

**Resultado:** no hay filas duplicadas en ninguna tabla.
Los nulos están solo en columnas descriptivas opcionales (`Company`, `State`,
`Fax`, `PostalCode`, `Phone`, `BillingState`, `BillingPostalCode`).
Ninguna columna que usaremos para los cálculos (`Total`, `CustomerId`,
`BillingCountry`, `InvoiceDate`) tiene nulos, así que **no se elimina ni se
imputa nada**.

## 3. Tipos de dato — convertir `InvoiceDate`

`read_csv` no interpreta fechas automáticamente: `InvoiceDate` llega como
`object` (texto). La convertimos con `pd.to_datetime()` para poder extraer año
y mes más adelante.

In [ ]:
print("Antes:", invoice["InvoiceDate"].dtype)

invoice["InvoiceDate"] = pd.to_datetime(invoice["InvoiceDate"])

print("Despues:", invoice["InvoiceDate"].dtype)
print("Rango de fechas:", invoice["InvoiceDate"].min(), "->", invoice["InvoiceDate"].max())

## 4. Top 10 países por ingresos

Agrupamos por `BillingCountry`, sumamos `Total` y ordenamos de mayor a menor.
Luego calculamos qué porcentaje del ingreso global aporta el primer país.

In [ ]:
ingresos_pais = invoice.groupby("BillingCountry")["Total"].sum().sort_values(ascending=False)

top10 = ingresos_pais.head(10)
print(top10)

In [ ]:
total_global = ingresos_pais.sum()
lider = top10.index[0]
pct = top10.iloc[0] / total_global * 100

print(f"Ingreso global: {total_global:,.2f}")
print(f"Pais lider: {lider} -> {top10.iloc[0]:,.2f} ({pct:.2f}% del total)")

**Lectura:** EE.UU. encabeza la lista, pero con un porcentaje relativamente bajo
del total: el ingreso está **repartido entre muchos países**, no concentrado en uno.

## 5. Ticket promedio global y comparación EE.UU. vs Francia

El *ticket promedio* es el monto promedio por factura: `Total.mean()`.

In [ ]:
ticket_global = invoice["Total"].mean()
print(f"Ticket promedio global: {ticket_global:.2f}")

In [ ]:
ticket_usa    = invoice.loc[invoice["BillingCountry"] == "USA",    "Total"].mean()
ticket_francia = invoice.loc[invoice["BillingCountry"] == "France", "Total"].mean()

print(f"USA    : {ticket_usa:.2f}")
print(f"France : {ticket_francia:.2f}")
print(f"Diferencia (USA - France): {ticket_usa - ticket_francia:.2f}")

**Lectura:** los dos tickets promedio son muy parecidos entre sí y respecto al
global. EE.UU. no factura más *por compra*, sino que tiene **más clientes y más
facturas**; ahí está su ventaja en ingresos.

## 6. Tabla por país con `.agg()` y columnas nombradas

Una sola instrucción que devuelve ingresos totales, número de facturas y ticket
promedio usando *named aggregation* (`nombre=("columna", "funcion")`).

In [ ]:
resumen_pais = (
    invoice.groupby("BillingCountry")
           .agg(ingresos_totales=("Total", "sum"),
                num_facturas=("InvoiceId", "count"),
                ticket_promedio=("Total", "mean"))
           .sort_values("ingresos_totales", ascending=False)
           .round(2)
)

resumen_pais.head(10)

## 7. Merge y comparación `Country` vs `BillingCountry`

Unimos las dos tablas por `CustomerId` y comparamos el país registrado del
cliente contra el país de facturación.

In [ ]:
df = invoice.merge(customer, on="CustomerId", how="inner")
print("Filas tras el merge:", df.shape)

discrepancias = df[df["BillingCountry"] != df["Country"]]
print("Facturas con pais distinto al del cliente:", len(discrepancias))

discrepancias[["InvoiceId", "CustomerId", "Country", "BillingCountry"]]

**Resultado:** 0 discrepancias. En las 412 facturas el país de facturación
coincide siempre con el país del cliente, es decir, **la dirección de facturación
se copia del perfil del cliente**. Esto valida la consistencia de las dos tablas.

## 8. Clientes sin facturas

Aquí el tipo de unión correcto es **LEFT JOIN** desde `Customer` hacia `Invoice`:
así conservamos a todos los clientes, incluso a los que no aparecen en `Invoice`.
Los que nunca facturaron quedan con `InvoiceId` nulo.

In [ ]:
left = customer.merge(invoice, on="CustomerId", how="left", indicator=True)

sin_factura = left[left["_merge"] == "left_only"]
print("Clientes sin ninguna factura:", sin_factura["CustomerId"].nunique())

sin_factura[["CustomerId", "FirstName", "LastName", "Country"]]

**Resultado:** ningún cliente quedó sin facturas — los 59 clientes de `Customer`
tienen al menos una compra. (`indicator=True` crea la columna `_merge`, que marca
`left_only` cuando no hubo coincidencia.)

## 9. Top 10 clientes por gasto total (con nombre completo)

Construimos el nombre completo concatenando `FirstName` y `LastName`, y sumamos
el `Total` de sus facturas.

In [ ]:
df["NombreCompleto"] = df["FirstName"] + " " + df["LastName"]

top_clientes = (
    df.groupby(["CustomerId", "NombreCompleto", "Country"])
      .agg(gasto_total=("Total", "sum"),
           num_facturas=("InvoiceId", "count"))
      .sort_values("gasto_total", ascending=False)
      .head(10)
      .round(2)
      .reset_index()
)

top_clientes

## 10. Tabla pivote: año (filas) × país (columnas)

Extraemos el año de `InvoiceDate` (ya convertido en el punto 3) con el accesor
`.dt.year` y armamos el pivote con la suma de `Total`.

In [ ]:
invoice["Anio"] = invoice["InvoiceDate"].dt.year

pivote = pd.pivot_table(invoice,
                        index="Anio",
                        columns="BillingCountry",
                        values="Total",
                        aggfunc="sum",
                        fill_value=0).round(2)

pivote

In [ ]:
# Version resumida: solo los 8 paises de mayor ingreso, mas facil de leer
top8 = ingresos_pais.head(8).index
pivote[top8]

## 11. Tasa de recompra

Porcentaje de clientes con más de una factura.

In [ ]:
facturas_por_cliente = invoice.groupby("CustomerId")["InvoiceId"].count()

recompradores = (facturas_por_cliente > 1).sum()
total_clientes = customer["CustomerId"].nunique()
tasa = recompradores / total_clientes * 100

print(f"Clientes totales        : {total_clientes}")
print(f"Clientes con >1 factura : {recompradores}")
print(f"Tasa de recompra        : {tasa:.2f}%")
print(f"Facturas por cliente (promedio): {facturas_por_cliente.mean():.1f}")

**Resultado:** 100%. Todos los clientes compraron más de una vez (en promedio
≈7 facturas cada uno). El negocio **no tiene problema de retención**; el reto
está en el monto por compra, no en que el cliente vuelva.

# 12. Reportes libres

Cuatro análisis adicionales sobre el dataset limpio. Cada uno indica **qué se
encontró**, **cómo se comprobó** y la **recomendación de negocio**.

### Reporte 1 — Concentración de ingresos (¿existe riesgo de dependencia?)

Pregunta: ¿el negocio depende de unos pocos clientes grandes?

In [ ]:
gasto = df.groupby("NombreCompleto")["Total"].sum().sort_values(ascending=False)

share_top1  = gasto.iloc[0]  / gasto.sum() * 100
share_top10 = gasto.head(10).sum() / gasto.sum() * 100

print(f"Cliente #1 aporta      : {share_top1:.2f}% del ingreso")
print(f"Top 10 clientes aportan: {share_top10:.2f}% del ingreso")
print(f"Gasto max: {gasto.max():.2f} | min: {gasto.min():.2f} | promedio: {gasto.mean():.2f}")

**Qué se encontró:** los 10 mejores clientes (17% de la base) generan apenas
~19% del ingreso, y el mejor cliente solo ~2%. Muy lejos del clásico 80/20.

**Cómo se comprobó:** suma de `Total` por cliente dividida entre el ingreso
global; se contrastaron máximo, mínimo y promedio para ver la dispersión, que
resultó baja.

**Recomendación:** no hay riesgo de dependencia de clientes clave, así que
**no conviene invertir en un programa VIP**. Como todos gastan parecido, el
crecimiento debe venir de subir el ticket promedio de *toda* la base
(bundles, precios por volumen) o de captar clientes nuevos.

### Reporte 2 — Evolución anual de ingresos

Pregunta: ¿el negocio está creciendo, estancado o cayendo?

In [ ]:
anual = (invoice.groupby("Anio")
                .agg(ingresos=("Total", "sum"),
                     facturas=("InvoiceId", "count"),
                     ticket=("Total", "mean"))
                .round(2))

anual["var_%"] = (anual["ingresos"].pct_change() * 100).round(2)
anual

**Qué se encontró:** los ingresos se mueven en una banda muy estrecha durante
los 5 años (≈449–481 por año), con variaciones interanuales de apenas unos
puntos porcentuales y sin tendencia clara. El negocio está **plano**.

**Cómo se comprobó:** agregación anual de ingresos, facturas y ticket, más
`pct_change()` para medir el crecimiento año contra año. Tanto el número de
facturas como el ticket se mantienen estables, así que el estancamiento no lo
explica ninguna de las dos variables por separado.

**Recomendación:** la operación actual está en equilibrio y no crecerá sola.
Cualquier meta de crecimiento exige una **acción deliberada** (nuevos mercados,
catálogo o promociones), y con esta serie histórica tan estable, cualquier
cambio superior a ~±8% después de una campaña puede atribuirse con confianza
a esa campaña.

### Reporte 3 — Estacionalidad mensual

Pregunta: ¿hay meses fuertes y débiles que sirvan para planificar campañas?

In [ ]:
invoice["Mes"] = invoice["InvoiceDate"].dt.month

mensual = (invoice.groupby("Mes")
                  .agg(ingresos=("Total", "sum"),
                       facturas=("InvoiceId", "count"))
                  .round(2))

mensual["%_del_total"] = (mensual["ingresos"] / mensual["ingresos"].sum() * 100).round(2)
print(mensual)

print(f"\nMes mas alto: {mensual['ingresos'].idxmax()} | Mes mas bajo: {mensual['ingresos'].idxmin()}")
print(f"Brecha alto vs bajo: {mensual['ingresos'].max() / mensual['ingresos'].min():.2f}x")

**Qué se encontró:** los 12 meses aportan entre 8.0% y 8.6% del ingreso total y
el mejor mes solo supera al peor por 1.08x. **No hay estacionalidad relevante.**

**Cómo se comprobó:** se agruparon los 5 años por mes calendario y se calculó
la participación de cada mes; ningún mes se despega del 8.3% que correspondería
a un reparto perfectamente uniforme.

**Recomendación:** no gastar presupuesto en campañas de temporada — no hay un
pico natural que aprovechar ni un valle que rescatar. Conviene un
**presupuesto de marketing constante**, y usar los meses indistintamente para
hacer pruebas A/B de precios, ya que la estacionalidad no contamina la
comparación.

### Reporte 4 — Desempeño de los representantes de ventas

Pregunta: ¿los agentes de soporte (`SupportRepId`) rinden parejo?

In [ ]:
reps = (df.groupby("SupportRepId")
          .agg(ingresos=("Total", "sum"),
               clientes=("CustomerId", "nunique"),
               facturas=("InvoiceId", "count"))
          .round(2))

reps["ingreso_x_cliente"] = (reps["ingresos"] / reps["clientes"]).round(2)
reps.sort_values("ingresos", ascending=False)

**Qué se encontró:** hay diferencias en el ingreso total por representante,
pero se explican casi por completo por **cuántos clientes tiene asignados cada
uno**: al normalizar (`ingreso_x_cliente`) los tres quedan prácticamente iguales.

**Cómo se comprobó:** se compararon ingresos absolutos contra ingresos por
cliente; el ranking absoluto y el número de clientes asignados siguen el mismo
orden, mientras que el ingreso por cliente se aplana.

**Recomendación:** **no premiar ni penalizar por ingreso absoluto**, porque
mide el tamaño de la cartera y no el desempeño. La métrica justa de comisión es
el ingreso por cliente asignado; y si se quiere subir la facturación de un
representante, la palanca real es **reasignar cartera**, no exigir más ventas.

---
## Conclusión general

El dataset está **limpio y consistente**: sin duplicados, sin clientes huérfanos
y con el país de facturación siempre alineado al del cliente. El negocio es
**estable pero plano**: ingresos sin crecimiento en 5 años, sin estacionalidad,
sin concentración de clientes y con 100% de recompra. La palanca de crecimiento
no es la retención (ya está al máximo) sino **subir el ticket promedio** y
**ampliar la base de clientes**.